In [ ]:
!pip install Unidecode

In [ ]:
!pip install wordcloud

In [ ]:
!pip install lime

In [ ]:
!pip install transformers

In [5]:
import warnings
warnings.filterwarnings("ignore")

In [6]:
# text preprocessing libraries
import re
import nltk
import pandas as pd
import numpy as np

from string import punctuation
from unidecode import unidecode
from nltk.corpus import stopwords
from nltk import sent_tokenize, word_tokenize

In [7]:
import matplotlib.pyplot as plt

from wordcloud import WordCloud

In [8]:
# text classification libraries
import seaborn as sns

from lime.lime_text import LimeTextExplainer
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import RandomUnderSampler
from transformers import BertTokenizer, BertForSequenceClassification

# Loading data

In [ ]:
url = ''
file_id = url.split('/')[-2]
read_url='https://drive.google.com/uc?id=' + file_id

data_set = pd.read_excel(read_url, index_col=False)

condition = [
    data_set["rotulo_humano"] == "sem_sintoma", # 0
    data_set["rotulo_humano"] == "sintoma" # 1
    # data_set["rotulo_vader"] == "positivo", # 2
]

values = [0, 1]

data_set["classification"] = np.select(condition, values)

data_set

# Data pre-processing

In [ ]:
nltk.download("rslp")
nltk.download("stopwords")
stopwords_list = stopwords.words("portuguese")
print(stopwords_list)

In [ ]:
data_process = data_set.copy()
data_process.columns

In [ ]:
old_texts = data_process["Texto"]
new_texts = []

for text in old_texts:
  text = str(text).lower()
  text = re.sub('@[^\s]+', '', text)
  text = unidecode(text)
  text = re.sub('<[^<]+?>','', text)
  text = ''.join(c for c in text if not c.isdigit())
  text = re.sub('((www\.[^\s]+)|(https?://[^\s]+)|(http?://[^\s]+))', '', text)
  text = ''.join(c for c in text if c not in punctuation)
  text = ' '.join([word for word in text.split() if word not in stopwords_list])
  text = ''.join(text.replace("\"", ""))
  text = ''.join(text.replace("'", ""))
  new_texts.append(text)

data_process["Texto"] = new_texts
data_process.head()


# Dataset balancing

In [ ]:
sns.countplot(x = data_process['rotulo_humano'])

In [ ]:
rus = RandomUnderSampler(random_state=0)
data_process["target"] = data_process["classification"]
X_bal, Y_bal = rus.fit_resample(data_process[['Texto']], data_process['classification'])
sns.countplot(x=Y_bal)

# Words cloud

In [ ]:
new_texts = data_process["Texto"]
all_words = ' '.join([text for text in new_texts])
word_cloud = WordCloud(width= 800, height= 500, max_font_size = 110, background_color="white", collocations = False).generate(all_words)
plt.figure(figsize=(20,10))
plt.imshow(word_cloud, interpolation='bilinear')
plt.axis("off")
plt.show()

# Cross validation: K-fold

In [16]:
train_df, test_df, train_label, test_label = train_test_split(X_bal, Y_bal, test_size=0.20, random_state=42)

In [17]:
train_df, valid_df, train_label, valid_label = train_test_split(train_df, train_label, test_size=0.20, random_state=42)

# Tokenization

Tokenization is a process for spliting raw texts into tokens, and encoding the tokens into numeric data.

To do this, we first initialize a ```BertTokenizer```

In [ ]:
bert_large = 'neuralmind/bert-large-portuguese-cased'

PRETRAINED_LM = bert_large
tokenizer = BertTokenizer.from_pretrained(bert_large, do_lower_case=True)
tokenizer

Define a function for encoding:

In [19]:
def encode(docs):
    '''
    This function takes list of texts and returns input_ids and attention_mask of texts
    '''
    encoded_dict = tokenizer.batch_encode_plus(docs, add_special_tokens=True, max_length=128, padding='max_length',
                            return_attention_mask=True, truncation=True, return_tensors='pt')
    input_ids = encoded_dict['input_ids']
    attention_masks = encoded_dict['attention_mask']
    return input_ids, attention_masks

Use the ```encode``` functionto get input ids and attention masks of the datasets:

In [20]:
train_input_ids, train_att_masks = encode(train_df['Texto'].values.tolist())
valid_input_ids, valid_att_masks = encode(valid_df['Texto'].values.tolist())
test_input_ids, test_att_masks = encode(test_df['Texto'].values.tolist())

# Creating ```Datasets``` and ```DataLoaders```

We'll use pytorch ```Dataset``` and ```DataLoader``` to split data into batches. For more detatils, you can check out another post on DataLoader.


Turn the labels into tensors:

In [ ]:
import torch
train_label.values
train_y = torch.LongTensor(train_label.values)
valid_y = torch.LongTensor(valid_label.values)
test_y = torch.LongTensor(test_label.values)
train_y.size(),valid_y.size(),test_y.size()

Create dataloaders for training

In [22]:
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

BATCH_SIZE = 16
train_dataset = TensorDataset(train_input_ids, train_att_masks, train_y)
train_sampler = RandomSampler(train_dataset)
train_dataloader = DataLoader(train_dataset, sampler=train_sampler, batch_size=BATCH_SIZE)

valid_dataset = TensorDataset(valid_input_ids, valid_att_masks, valid_y)
valid_sampler = SequentialSampler(valid_dataset)
valid_dataloader = DataLoader(valid_dataset, sampler=valid_sampler, batch_size=BATCH_SIZE)

test_dataset = TensorDataset(test_input_ids, test_att_masks, test_y)
test_sampler = SequentialSampler(test_dataset)
test_dataloader = DataLoader(test_dataset, sampler=test_sampler, batch_size=BATCH_SIZE)

## Bert For Sequence Classification Model

We will initiate the  `BertForSequenceClassification ` model from Huggingface, which allows easily fine-tuning the pretrained BERT mode for classification task.


You will see a warning that some parts of the model are randomly initialized. This is normal since the classification head has not yet been trained.

In [ ]:
train_label.unique()

In [ ]:
from transformers import BertForSequenceClassification
N_labels = len(train_label.unique())
model = BertForSequenceClassification.from_pretrained(PRETRAINED_LM,
                                                      num_labels=N_labels,
                                                      output_attentions=False,
                                                      output_hidden_states=False)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


In [26]:
model = model.to(device)

# Fine-tuning

## Optimizer and Scheduler

An **optimizer** is for tuning parameters in the model, which is set up with a learning rate.

Selection of the learning rate is important. In practice, it's common to use a **scheduler** to decrease the learning rate during training.

In [27]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# Best results: 07 and 08
EPOCHS = 5
LEARNING_RATE = 2e-6

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = get_linear_schedule_with_warmup(optimizer,
             num_warmup_steps=0,
            num_training_steps=len(train_dataloader)*EPOCHS )

### **Training Loop**

The training loop is where the magic of deep learning happens. The model will be fine-tuned on the emotion dataset for classification task.

In [ ]:
#collapse-output
from torch.nn.utils import clip_grad_norm_
from tqdm.notebook import tqdm
import numpy as np
import math

train_loss_per_epoch = []
val_loss_per_epoch = []

for epoch_num in range(EPOCHS):
    print('Epoch: ', epoch_num + 1)
    '''
    Training
    '''
    model.train()
    train_loss = 0
    for step_num, batch_data in enumerate(tqdm(train_dataloader, desc='Training')):
        input_ids, att_mask, labels = [data.to(device) for data in batch_data]

        output = model(input_ids=input_ids, attention_mask=att_mask, labels=labels)

        loss = output.loss
        train_loss += loss.item()

        model.zero_grad()
        loss.backward()
        del loss

        clip_grad_norm_(parameters=model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

    train_loss_per_epoch.append(train_loss / (step_num + 1))


    '''
    Validation
    '''
    model.eval()
    valid_loss = 0
    valid_pred = []
    with torch.no_grad():
        for step_num_e, batch_data in enumerate(tqdm(valid_dataloader, desc='Validation')):
            input_ids, att_mask, labels = [data.to(device) for data in batch_data]
            output = model(input_ids=input_ids, attention_mask=att_mask, labels=labels)

            loss = output.loss
            valid_loss += loss.item()

            valid_pred.append(np.argmax(output.logits.cpu().detach().numpy(), axis=-1))

    val_loss_per_epoch.append(valid_loss / (step_num_e + 1))
    valid_pred = np.concatenate(valid_pred)

    '''
    Loss message
    '''
    print("{0}/{1} train loss: {2} ".format(step_num+1, math.ceil(len(train_df) / BATCH_SIZE), train_loss / (step_num + 1)))
    print("{0}/{1} val loss: {2} ".format(step_num_e+1, math.ceil(len(valid_df) / BATCH_SIZE), valid_loss / (step_num_e + 1)))

You can see in the output that the training and validation losses steadily decreases in each epoch.

In [ ]:
from matplotlib import pyplot as plt
epochs = range(1, EPOCHS+1)
fig, ax = plt.subplots()
ax.plot(epochs,train_loss_per_epoch,label ='training loss')
ax.plot(epochs, val_loss_per_epoch, label = 'validation loss' )
ax.set_title('Training and Validation loss')
ax.set_xlabel('Epochs')
ax.set_ylabel('Loss')
ax.legend()
plt.show()

## Performance Metrics
It's common to use precision, recall, and F1-score as the performance metrics.

In [ ]:
label_names = ['sem sintoma', 'sintoma']
label_names

In [ ]:
from sklearn.metrics import classification_report
print('classifiation report')
print(classification_report(valid_pred, valid_label.to_numpy(), target_names=label_names))

## Error Analysis
With the predictions, we can plot the confusion matrix:

In [32]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
def plot_confusion_matrix(y_preds, y_true, labels=None):
  cm = confusion_matrix(y_true, y_preds, normalize="true")
  fig, ax = plt.subplots(figsize=(6, 6))
  disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
  disp.plot(cmap="Blues", values_format=".2f", ax=ax, colorbar=False)
  plt.title("Normalized confusion matrix")
  plt.show()

In [ ]:
plot_confusion_matrix(valid_pred,valid_label.to_numpy(),labels=label_names)

## Prediction
Now let's use the trained model to predict the testing set.

In [ ]:
model.eval()
test_pred = []
test_loss= 0
with torch.no_grad():
    for step_num, batch_data in tqdm(enumerate(test_dataloader)):
        input_ids, att_mask, labels = [data.to(device) for data in batch_data]
        output = model(input_ids = input_ids, attention_mask=att_mask, labels= labels)

        loss = output.loss
        test_loss += loss.item()

        test_pred.append(np.argmax(output.logits.cpu().detach().numpy(),axis=-1))
test_pred = np.concatenate(test_pred)

In [ ]:
print('classifiation report')
print(classification_report(test_pred, test_label.to_numpy(),target_names=label_names))

With the predictions, we can plot the confusion matrix again:

In [ ]:
plot_confusion_matrix(test_pred,test_label.to_numpy(),labels=label_names)

Output the misclassified text:

In [ ]:
test_df['pred'] = test_pred
test_df['label'] = test_label
test_df.reset_index(level=0)
print(test_df[test_df['label']!=test_df['pred']].shape)
test_df[test_df['label']!=test_df['pred']][['Texto','label','pred']].head(10)

## **Salvando os resultados e o modelo binário do BERT**

In [38]:
import shutil

In [ ]:
NAME_MODEL = 'BERT-Large'

test_df.to_csv(f"test_results_{NAME_MODEL}.csv", index=False)
shutil.copy(f'/content/test_results_{NAME_MODEL}.csv', f'/content/drive/MyDrive/test_results_{NAME_MODEL}')

MODEL_PATH = NAME_MODEL + ".bin"
torch.save(model.state_dict(), MODEL_PATH)

shutil.copy(f'/content/{MODEL_PATH}', f'/content/drive/MyDrive/{MODEL_PATH}')